# TF-IDF

This notebook covers TF-IDF (Term Frequency-Inverse Document Frequency), a powerful text weighting scheme used in information retrieval and text mining.

TF-IDF adjusts word frequencies based on how rare or common they are across documents, making it more sophisticated than simple word counts. This technique is widely used for document similarity, search engine ranking, and as features for machine learning models.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import math

print("Libraries imported successfully.")

## 1. Understanding TF-IDF

TF-IDF consists of two components:

**Term Frequency (TF)**: How often a term appears in a document
- TF(t,d) = (Number of times term t appears in document d) / (Total number of terms in document d)

**Inverse Document Frequency (IDF)**: How rare or common a term is across all documents
- IDF(t) = log(Total number of documents / Number of documents containing term t)

**TF-IDF Score**: TF(t,d) × IDF(t)

High TF-IDF scores indicate terms that are important within a specific document but rare across the corpus.

In [ ]:
# Sample corpus for demonstration
corpus = [
    "This is the first document. First document is about machine learning.",
    "This document is the second document. Second document covers deep learning.",
    "And this is the third one. Third document discusses neural networks.",
    "Is this the first document? First document focuses on NLP techniques.",
]

print("=== Sample Corpus ===")
for i, doc in enumerate(corpus, 1):
    print(f"Document {i}: {doc}\n")

In [ ]:
# Manual TF-IDF implementation
def compute_tf(text):
    """Compute Term Frequency for a document."""
    words = text.lower().split()
    words = [word.rstrip('.,!?;') for word in words]
    total_words = len(words)
    word_counts = Counter(words)
    
    # Calculate TF for each word
    tf = {word: count / total_words for word, count in word_counts.items()}
    return tf

def compute_idf(corpus):
    """Compute Inverse Document Frequency for all terms."""
    num_docs = len(corpus)
    
    # Get all unique words
    all_words = set()
    for doc in corpus:
        words = doc.lower().split()
        words = [word.rstrip('.,!?;') for word in words]
        all_words.update(words)
    
    # Count documents containing each word
    doc_counts = {word: 0 for word in all_words}
    for doc in corpus:
        words = set(doc.lower().split())
        words = {word.rstrip('.,!?;') for word in words}
        for word in words:
            if word in doc_counts:
                doc_counts[word] += 1
    
    # Calculate IDF
    idf = {word: math.log(num_docs / count) for word, count in doc_counts.items()}
    return idf

def compute_tfidf_manual(text, idf):
    """Compute TF-IDF for a document."""
    tf = compute_tf(text)
    tfidf = {word: tf_val * idf[word] for word, tf_val in tf.items()}
    return tfidf

# Compute IDF for the corpus
idf = compute_idf(corpus)

print("=== IDF Values ===")
for word, value in sorted(idf.items()):
    print(f"  {word}: {value:.4f}")

In [ ]:
# Compute TF-IDF for each document
print("=== TF-IDF Scores (Manual Implementation) ===\n")
for i, doc in enumerate(corpus, 1):
    tfidf = compute_tfidf_manual(doc, idf)
    print(f"Document {i} - Top 5 terms by TF-IDF:")
    sorted_terms = sorted(tfidf.items(), key=lambda x: x[1], reverse=True)[:5]
    for word, score in sorted_terms:
        print(f"  {word}: {score:.4f}")
    print()

## 2. Using Scikit-learn's TfidfVectorizer

Scikit-learn provides the TfidfVectorizer which implements TF-IDF with various options and proper normalisation.

In [ ]:
# Using TfidfVectorizer from scikit-learn
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print("=== Using TfidfVectorizer ===")
print(f"Vocabulary: {tfidf_vectorizer.get_feature_names_out()}")
print(f"\nTF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"\nTF-IDF Matrix (dense):\n{tfidf_matrix.toarray()}")

In [ ]:
# Convert to DataFrame for better visualisation
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("=== TF-IDF DataFrame ===")
print(tfidf_df.round(4))

## 3. Understanding the Differences: BoW vs TF-IDF

Let's compare Bag of Words and TF-IDF to understand why TF-IDF often produces better results.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Create both representations
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(corpus)

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print("=== Comparison: Bag of Words vs TF-IDF ===\n")

# Show specific term comparison
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Document 1 - Term comparison:")
print(f"{'Term':<15} {'BoW':<10} {'TF-IDF':<10}")
print("-" * 35)
for i, term in enumerate(feature_names):
    bow_val = bow_matrix[0, i]
    tfidf_val = tfidf_matrix[0, i]
    if bow_val > 0 or tfidf_val > 0:
        print(f"{term:<15} {bow_val:<10} {tfidf_val:.4f}")

## 4. Customising TfidfVectorizer

The TfidfVectorizer offers many options to fine-tune the TF-IDF representation.

In [ ]:
print("=== Customising TfidfVectorizer ===\n")

# Example 1: With stop words removal
tfidf_no_stop = TfidfVectorizer(stop_words='english')
matrix_no_stop = tfidf_no_stop.fit_transform(corpus)
print("1. With stop words removed:")
print(f"   Vocabulary: {tfidf_no_stop.get_feature_names_out()}\n")

# Example 2: With sublinear TF (uses 1 + log(tf) instead of raw counts)
tfidf_sublinear = TfidfVectorizer(sublinear_tf=True)
matrix_sublinear = tfidf_sublinear.fit_transform(corpus)
print("2. With sublinear_tf=True (uses 1 + log(tf)):")
print(f"   Document 1, 'document' value: {matrix_sublinear[0, feature_names.tolist().index('document')]:.4f}\n")

# Example 3: With custom n-gram range
tfidf_ngram = TfidfVectorizer(ngram_range=(1, 2))
matrix_ngram = tfidf_ngram.fit_transform(corpus)
print("3. With bigrams (ngram_range=(1, 2)):")
print(f"   Number of features: {len(tfidf_ngram.get_feature_names_out())}")

## 5. Practical Application: Document Similarity

One common use of TF-IDF is finding similar documents using cosine similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity between all documents
similarity_matrix = cosine_similarity(tfidf_matrix)

print("=== Document Similarity Matrix (Cosine Similarity) ===\n")
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"Doc {i+1}" for i in range(len(corpus))],
    columns=[f"Doc {i+1}" for i in range(len(corpus))]
)
print(similarity_df.round(4))

print("\n=== Most Similar Document Pairs ===")
for i in range(len(corpus)):
    for j in range(i+1, len(corpus)):
        sim = similarity_matrix[i, j]
        if sim > 0.1:  # Only show non-zero similarities
            print(f"  Document {i+1} and Document {j+1}: {sim:.4f}")

## 6. Practical Application: Keyword Extraction

TF-IDF can be used to extract important keywords from documents.

In [ ]:
def extract_keywords_tfidf(documents, top_n=5):
    """Extract top keywords from each document using TF-IDF."""
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(documents)
    feature_names = vectorizer.get_feature_names_out()
    
    keywords = []
    for i, doc in enumerate(documents):
        # Get TF-IDF scores for this document
        doc_vector = tfidf_matrix[i].toarray().flatten()
        
        # Get top N indices
        top_indices = doc_vector.argsort()[-top_n:][::-1]
        
        # Get top keywords and scores
        doc_keywords = [(feature_names[idx], doc_vector[idx]) for idx in top_indices if doc_vector[idx] > 0]
        keywords.append(doc_keywords)
    
    return keywords

# Extract keywords
keywords = extract_keywords_tfidf(corpus, top_n=3)

print("=== Extracted Keywords (Top 3 per Document) ===\n")
for i, doc_keywords in enumerate(keywords, 1):
    print(f"Document {i}:")
    for keyword, score in doc_keywords:
        print(f"  {keyword}: {score:.4f}")
    print()

## Summary

In this notebook, we covered TF-IDF (Term Frequency-Inverse Document Frequency):

1. **TF-IDF Theory**: Understood the mathematical foundations of TF and IDF
2. **Manual Implementation**: Built TF-IDF from scratch to understand the concept
3. **TfidfVectorizer**: Used scikit-learn's implementation with various options
4. **BoW vs TF-IDF**: Compared the two approaches to understand the differences
5. **Customisation**: Explored options like sublinear_tf and n-gram ranges
6. **Document Similarity**: Applied TF-IDF for finding similar documents
7. **Keyword Extraction**: Used TF-IDF for extracting important keywords

TF-IDF is a powerful technique that forms the basis for many NLP applications. While more sophisticated methods like word embeddings and transformers have emerged, TF-IDF remains useful for its simplicity and interpretability.